<a href="https://colab.research.google.com/github/BhumiPatil1507/YM_functionable_code/blob/main/Feature_Extraction_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Feature Extraction code

In [ ]:
%%writefile /content/extract_features.py
"""
extract_features.py — chunked RDKit feature extraction compatible with batch_feat_ext.py.

This is a newly written implementation, not the unavailable original extractor.
It creates four aligned CSVs and retains source columns (including YM labels).

Outputs:
  {stem}_fingerprints.csv  : 2048 Morgan/ECFP4 bits (default)
  {stem}_descriptors.csv   : 16 RDKit descriptors
  {stem}_graph.csv         : 17 graph/topology features
  {stem}_funcgroups.csv    : 33 SMARTS indicators

The exact 16/17/33 definitions are newly specified here; they may differ from
the original senior implementation even though the dimensions match the README.
"""

from __future__ import annotations
import argparse
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, Lipinski, rdFingerprintGenerator, rdMolDescriptors

RDLogger.DisableLog("rdApp.error")
RDLogger.DisableLog("rdApp.warning")

DESCRIPTOR_FUNCS = {
    "MolWt": Descriptors.MolWt,
    "HeavyAtomCount": Lipinski.HeavyAtomCount,
    "NumHAcceptors": Lipinski.NumHAcceptors,
    "NumHDonors": Lipinski.NumHDonors,
    "MolLogP": Descriptors.MolLogP,
    "TPSA": rdMolDescriptors.CalcTPSA,
    "NumRotatableBonds": Lipinski.NumRotatableBonds,
    "RingCount": Lipinski.RingCount,
    "NumAromaticRings": Lipinski.NumAromaticRings,
    "NumAliphaticRings": Lipinski.NumAliphaticRings,
    "FractionCSP3": rdMolDescriptors.CalcFractionCSP3,
    "LabuteASA": rdMolDescriptors.CalcLabuteASA,
    "BalabanJ": Descriptors.BalabanJ,
    "BertzCT": Descriptors.BertzCT,
    "HallKierAlpha": Descriptors.HallKierAlpha,
    "Chi0v": Descriptors.Chi0v,
}
DESCRIPTOR_NAMES = list(DESCRIPTOR_FUNCS)

GRAPH_NAMES = [
    "graph_num_atoms", "graph_num_bonds", "graph_num_heavy_atoms",
    "graph_num_wildcards", "graph_mean_degree", "graph_max_degree",
    "graph_min_degree", "graph_degree_std", "graph_num_components",
    "graph_num_rings", "graph_num_aromatic_atoms", "graph_num_aromatic_bonds",
    "graph_num_single_bonds", "graph_num_double_bonds", "graph_num_triple_bonds",
    "graph_cycle_rank", "graph_fraction_heavy_atoms_in_rings",
]

# 33 functional-group/substructure indicators. These are explicit new definitions.
FUNCTIONAL_GROUP_SMARTS = [
    ("fg_alcohol", "[OX2H][CX4]"),
    ("fg_phenol", "[OX2H]c"),
    ("fg_ether", "[OD2]([#6])[#6]"),
    ("fg_aldehyde", "[CX3H1](=O)[#6]"),
    ("fg_ketone", "[#6][CX3](=O)[#6]"),
    ("fg_carboxylic_acid", "[CX3](=O)[OX2H1]"),
    ("fg_ester", "[CX3](=O)[OX2H0][#6]"),
    ("fg_amide", "[NX3][CX3](=[OX1])[#6]"),
    ("fg_primary_amine", "[NX3;H2;!$(NC=O)]"),
    ("fg_secondary_amine", "[NX3;H1;!$(NC=O)]([#6])[#6]"),
    ("fg_tertiary_amine", "[NX3;H0;!$(NC=O)]([#6])([#6])[#6]"),
    ("fg_imine", "[CX3]=[NX2]"),
    ("fg_nitrile", "[CX2]#N"),
    ("fg_nitro", "[N+](=O)[O-]"),
    ("fg_nitro_alt", "N(=O)=O"),
    ("fg_azo", "[NX2]=[NX2]"),
    ("fg_aromatic_n", "[n]"),
    ("fg_pyridine_like_n", "[nH0]"),
    ("fg_aromatic_oxygen", "[o]"),
    ("fg_aromatic_sulfur", "[s]"),
    ("fg_fluorine", "[F]"),
    ("fg_chlorine", "[Cl]"),
    ("fg_bromine", "[Br]"),
    ("fg_iodine", "[I]"),
    ("fg_alkene", "[CX3]=[CX3]"),
    ("fg_alkyne", "[CX2]#[CX2]"),
    ("fg_aromatic_ring", "a1aaaaa1"),
    ("fg_epoxide_like", "[OX2r3]1[CX4r3][CX4r3]1"),
    ("fg_silicon", "[Si]"),
    ("fg_phosphorus", "[P]"),
    ("fg_sulfonyl", "[SX4](=O)(=O)"),
    ("fg_sulfide", "[#16X2]([#6])[#6]"),
    ("fg_carbonate", "[OX2][CX3](=O)[OX2]"),
]
_COMPILED_FG = []
for _name, _smarts in FUNCTIONAL_GROUP_SMARTS:
    _query = Chem.MolFromSmarts(_smarts)
    if _query is None:
        raise ValueError(f"Invalid functional-group SMARTS: {_name}: {_smarts}")
    _COMPILED_FG.append((_name, _query))
FG_NAMES = [name for name, _ in FUNCTIONAL_GROUP_SMARTS]


def _find_smiles_column(columns):
    candidates = ("smiles", "SMILES", "canonical_smiles", "CanonicalSMILES")
    for candidate in candidates:
        if candidate in columns:
            return candidate
    by_lower = {str(c).strip().lower(): c for c in columns}
    for candidate in ("smiles", "canonical_smiles"):
        if candidate in by_lower:
            return by_lower[candidate]
    raise ValueError(f"No SMILES column found. Columns: {list(columns)}")


def _descriptor_values(mol):
    out = {}
    for name, fn in DESCRIPTOR_FUNCS.items():
        try:
            value = float(fn(mol))
            out[name] = value if np.isfinite(value) else np.nan
        except Exception:
            out[name] = np.nan
    return out


def _graph_values(mol):
    n_atoms, n_bonds, heavy = mol.GetNumAtoms(), mol.GetNumBonds(), mol.GetNumHeavyAtoms()
    degrees = np.asarray([a.GetDegree() for a in mol.GetAtoms()], dtype=float)
    if not degrees.size:
        degrees = np.array([0.0])
    components = len(Chem.GetMolFrags(mol))
    heavy_in_rings = sum(a.GetAtomicNum() > 1 and a.IsInRing() for a in mol.GetAtoms())
    return {
        "graph_num_atoms": float(n_atoms),
        "graph_num_bonds": float(n_bonds),
        "graph_num_heavy_atoms": float(heavy),
        "graph_num_wildcards": float(sum(a.GetAtomicNum() == 0 for a in mol.GetAtoms())),
        "graph_mean_degree": float(degrees.mean()),
        "graph_max_degree": float(degrees.max()),
        "graph_min_degree": float(degrees.min()),
        "graph_degree_std": float(degrees.std()),
        "graph_num_components": float(components),
        "graph_num_rings": float(rdMolDescriptors.CalcNumRings(mol)),
        "graph_num_aromatic_atoms": float(sum(a.GetIsAromatic() for a in mol.GetAtoms())),
        "graph_num_aromatic_bonds": float(sum(b.GetIsAromatic() for b in mol.GetBonds())),
        "graph_num_single_bonds": float(sum(b.GetBondType() == Chem.BondType.SINGLE for b in mol.GetBonds())),
        "graph_num_double_bonds": float(sum(b.GetBondType() == Chem.BondType.DOUBLE for b in mol.GetBonds())),
        "graph_num_triple_bonds": float(sum(b.GetBondType() == Chem.BondType.TRIPLE for b in mol.GetBonds())),
        "graph_cycle_rank": float(max(0, n_bonds - n_atoms + components)),
        "graph_fraction_heavy_atoms_in_rings": float(heavy_in_rings / heavy) if heavy else 0.0,
    }


def _fingerprint_values(mol, generator, nbits):
    fp = generator.GetFingerprint(mol)
    arr = np.zeros((nbits,), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return {f"fp_{i}": int(arr[i]) for i in range(nbits)}


def process_csv_file(csv_path, output_dir, stem, morgan_radius=2,
                     morgan_nbits=2048, chunksize=1000):
    """Function signature and returned stats match batch_feat_ext.py."""
    csv_path, output_dir = Path(csv_path), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    header = pd.read_csv(csv_path, nrows=0)
    smiles_col = _find_smiles_column(header.columns)
    # Avoid accidentally carrying old feature columns into regenerated outputs.
    base_cols = [
        c for c in header.columns
        if not (str(c).startswith(("fp_", "graph_", "fg_"))
                or str(c) in DESCRIPTOR_NAMES)
    ]
    output_paths = {
        ft: output_dir / f"{stem}_{ft}.csv"
        for ft in ("descriptors", "graph", "funcgroups", "fingerprints")
    }
    # New run: overwrite all four outputs. Process in chunks to avoid retaining
    # 500k x 2048 Python objects in RAM.
    for path in output_paths.values():
        if path.exists():
            path.unlink()

    generator = rdFingerprintGenerator.GetMorganGenerator(
        radius=int(morgan_radius), fpSize=int(morgan_nbits), includeChirality=False
    )
    total_rows = total_success = total_failed = 0
    start = time.time()

    for chunk in pd.read_csv(csv_path, chunksize=int(chunksize)):
        # Keep source columns and order; all four output files use the same rows.
        base = chunk[base_cols].reset_index(drop=True)
        smiles_values = chunk[smiles_col].fillna("").astype(str).tolist()
        desc_rows, graph_rows, fg_rows, fp_rows = [], [], [], []
        for smiles in smiles_values:
            mol = None
            if smiles.strip():
                try:
                    mol = Chem.MolFromSmiles(smiles)
                except Exception:
                    mol = None
            if mol is None:
                total_failed += 1
                desc_rows.append({k: np.nan for k in DESCRIPTOR_NAMES})
                graph_rows.append({k: np.nan for k in GRAPH_NAMES})
                fg_rows.append({k: np.nan for k in FG_NAMES})
                fp_rows.append({f"fp_{j}": np.nan for j in range(int(morgan_nbits))})
            else:
                total_success += 1
                desc_rows.append(_descriptor_values(mol))
                graph_rows.append(_graph_values(mol))
                fg_rows.append({name: float(mol.HasSubstructMatch(q)) for name, q in _COMPILED_FG})
                fp_rows.append(_fingerprint_values(mol, generator, int(morgan_nbits)))

        feature_frames = {
            "descriptors": pd.DataFrame(desc_rows, columns=DESCRIPTOR_NAMES),
            "graph": pd.DataFrame(graph_rows, columns=GRAPH_NAMES),
            "funcgroups": pd.DataFrame(fg_rows, columns=FG_NAMES),
            "fingerprints": pd.DataFrame(fp_rows, columns=[f"fp_{j}" for j in range(int(morgan_nbits))]),
        }
        for ft, feature_frame in feature_frames.items():
            out = pd.concat([base, feature_frame], axis=1)
            out.to_csv(output_paths[ft], mode="a", header=not output_paths[ft].exists(), index=False)

        total_rows += len(chunk)
        elapsed = max(time.time() - start, 1e-9)
        rate = total_rows / elapsed
        eta_min = max(0, (sum(1 for _ in []) or 0))  # placeholder; exact total needs a full scan
        print(
            f"[{stem}] processed {total_rows:,} rows | "
            f"valid {total_success:,} | invalid {total_failed:,} | "
            f"{rate:.1f} rows/s | elapsed {elapsed/60:.1f} min",
            flush=True,
        )

    for ft, path in output_paths.items():
        print(f"[SAVED] {path} ({path.stat().st_size / (1024**3):.2f} GiB)")
    return {
        "rows_processed": int(total_rows),
        "rows_success": int(total_success),
        "rows_failed": int(total_failed),
    }


def main():
    parser = argparse.ArgumentParser(description="Extract polymer RDKit features.")
    parser.add_argument("--input", required=True)
    parser.add_argument("--output", required=True)
    parser.add_argument("--dataset", default=None)
    parser.add_argument("--morgan-radius", type=int, default=2)
    parser.add_argument("--morgan-nbits", type=int, default=2048)
    parser.add_argument("--chunksize", type=int, default=1000)
    args = parser.parse_args()
    input_path = Path(args.input)
    stem = args.dataset or input_path.stem
    stats = process_csv_file(
        input_path, Path(args.output), stem,
        morgan_radius=args.morgan_radius,
        morgan_nbits=args.morgan_nbits,
        chunksize=args.chunksize,
    )
    print("\nExtraction summary:")
    for key, value in stats.items():
        print(f"  {key}: {value:,}")


if __name__ == "__main__":
    main()